In [ ]:
# Code describing the various ways that data can be loaded into the quantum environment
# Then, a quantum random walk algorithm is used to project values forward to illustrate uses
# 1) Angle Encoding
# Code and process developed by: Dr. Michael P. Haydock - IBM Fellow Emeritus, Visiting Professor at St. Olaf College
# Initial Coding: 3/4/2025

# Load the libraries
import pandas as pd
import numpy as np
import pennylane as qml

# Load data
file_path = "economic_data.csv"  # Replace with your file path
data = pd.read_csv(file_path)  # Read the CSV file into a pandas DataFrame

# Drop the 'Date' column and use only numerical columns
numeric_data = data.iloc[:, 1:]  # Select all columns to the right of 'Date'

# Normalize the data to range [0, π] for angle encoding
normalized_data = (numeric_data - numeric_data.min()) / (numeric_data.max() - numeric_data.min()) * np.pi

# Validate the normalized data
if normalized_data.isnull().values.any():  # Check for missing values
    raise ValueError("Unexpected NaN values after normalization. Check numeric ranges.")

# Dynamically set the number of qubits
num_qubits = len(numeric_data.columns)  # Number of qubits matches the number of input columns
dev = qml.device("default.qubit", wires=num_qubits)  # Define the quantum device

# Quantum angle encoding function
def angle_encoding(data):
    """
    Encode data into qubits using angle rotations:
    - Each value is encoded as an angle for RX gates.
    """
    for idx, value in enumerate(data):
        qml.RX(value, wires=idx)  # Apply an RX rotation to encode the value

# Define a single quantum random walk step with angle encoding
def random_walk_step():
    """
    Add variability and dynamics to the quantum walk:
    - Apply Hadamard gates for superposition.
    - Use parameterized RY and CRX gates with randomized angles.
    """
    for wire in range(num_qubits):
        qml.Hadamard(wires=wire)  # Superposition on each qubit
        qml.RY(np.random.uniform(0, np.pi), wires=wire)  # Randomize RY gate angles
    for i in range(num_qubits - 1):
        qml.CRX(np.random.uniform(0, np.pi / 2), wires=[i, i + 1])  # Controlled entanglement
    qml.RZ(np.random.uniform(0, np.pi), wires=num_qubits - 1)  # Add variation with RZ on the last qubit

# Quantum node for random walk simulation
@qml.qnode(dev)
def random_walk(data):
    """
    Perform a single step of the quantum random walk:
    - Encodes input data using angle encoding.
    - Applies random walk steps to generate dynamics.
    - Returns probabilities of all computational basis states.
    """
    angle_encoding(data)  # Encode the input data with angle rotations
    for _ in range(3):  # Apply the random walk step multiple times for richer dynamics
        random_walk_step()
    return qml.probs(wires=range(num_qubits))  # Return the probabilities of each state

# Forecast function for future time periods
def forecast(data, steps=12):
    """
    Generate forecasts for the next 'steps' time periods:
    - Starts with the last row of the input data.
    - Iteratively performs quantum random walk to predict values.
    - Rescales predictions back to the original range.
    """
    input_vector = data.iloc[-1].values  # Use the last row as the initial input
    predictions = []  # Initialize predictions list

    for _ in range(steps):  # Loop for the specified number of steps
        probabilities = random_walk(input_vector)  # Perform the quantum random walk
        next_values = probabilities[:len(data.columns)]  # Use only relevant probabilities
        # Rescale predictions back to the original data range
        next_values = next_values * (numeric_data.max() - numeric_data.min()) + numeric_data.min()
        predictions.append(next_values)  # Append predicted values to the list

        # Add slight noise to ensure variability in the input for the next step
        noise = np.random.uniform(-0.01, 0.01, size=next_values.shape)
        input_vector = ((next_values + noise) - numeric_data.min().values) / (numeric_data.max().values - numeric_data.min().values) * np.pi

    # Convert predictions into a DataFrame
    return pd.DataFrame(predictions, columns=numeric_data.columns)

# Generate the 12-step forecast
forecasted_data = forecast(normalized_data)  # Call the forecast function on the normalized data
forecasted_data.index = [f"Forecast {i+1}" for i in range(12)]  # Label rows as "Forecast 1", "Forecast 2", etc.

# Visualize the quantum circuit for the first input
input_vector = normalized_data.iloc[-1].values  # Use the last row of normalized data for visualization
drawer = qml.draw(random_walk)  # Prepare to draw the circuit
circuit_diagram = drawer(input_vector)  # Generate the circuit diagram
print("Quantum Circuit for the Angle Encoding Random Walk:")
print(circuit_diagram)  # Print the quantum circuit diagram

# Display the forecasted data
print("\n12-Time-Period Forecast:")
print(forecasted_data)  # Display forecasted data